<a href="https://colab.research.google.com/github/vieer-dwivedi/production-rag-pipeline-vivekananda/blob/main/SlideChunking/SlideChunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [ ]:
! pip install requests==2.32.4  PyMuPDF==1.27.2.3 faiss-cpu==1.14.3 sentence-transformers==5.6.0

# Load Pdf

In [ ]:
import fitz
import requests
remote = requests.get("https://d1.awsstatic.com/whitepapers/aws-overview.pdf")
doc = fitz.Document(
    stream=remote.content,
    filetype="pdf"
    )


# Document Structured Chunking

## Get Headings and paragraphs

In [ ]:
from collections import defaultdict
sections = defaultdict(list)
heading = None
for page in doc[10:]:
    for block in page.get_text("rawdict")["blocks"]:
        if block["type"] != 0:
            continue
        spans = [
            span
            for line in block["lines"]
            for span in line["spans"]
        ]
        text = "".join(
            char["c"]
            for span in spans
            for char in span["chars"]
        ).strip()
        if not text:
            continue
        size = max(span["size"] for span in spans)
        if any("Bold" in span["font"] for span in spans) and size >= 16:
            heading = text
        elif heading and size == 12 and len(text) > 50:
            sections[heading].append(text)

### Print Paragraphs

In [ ]:
for heading, paras in sections.items():

    print("\n" + "=" * 80)
    print(heading)

    for i, para in enumerate(paras, start=1):
        print(f"\nParagraph {i}")
        print(para[:200])

In [ ]:
for heading, paragraphs in sections.items():

    print(f"\n📁 {heading}")

    for i in range(len(paragraphs)):
        print(f"   ├── Paragraph {i+1}")

## Add meta data

In [ ]:
metadata = []
for page in doc[10:]:
    page_num = page.number + 1
    for block in page.get_text("rawdict")["blocks"]:
        if block["type"] != 0:
            continue
        spans = [
            span
            for line in block["lines"]
            for span in line["spans"]
        ]
        text = "".join(
            char["c"]
            for span in spans
            for char in span["chars"]
        ).strip()
        if not text:
            continue
        size = max(span["size"] for span in spans)
        if any("Bold" in span["font"] for span in spans) and size >= 16:
            heading = text
        elif heading and size == 12 and len(text) > 50:
            metadata.append(
                {
                    "content": text,
                    "source": "aws-overview.pdf",
                    "page": page_num,
                    "heading": heading,
                    "bbox": block["bbox"]
                }
            )

### Print Meta data per paragraphs

In [ ]:
from pprint import pprint

print(f"Total paragraphs: {len(metadata)}\n")

for i, para in enumerate(metadata[:10], start=1):
    print(f"\n{'='*80}")
    print(f"Paragraph {i}")
    print(f"{'='*80}")
    pprint(para)

# Slide Chunking

## Add Slide Chunking

In [ ]:
WINDOW_SIZE = 3
OVERLAP = 1
final_chunks = []
for heading, paragraphs in sections.items():
    step = WINDOW_SIZE - OVERLAP
    for i in range(0, len(paragraphs), step):
        window = paragraphs[i:i + WINDOW_SIZE]
        if not window:
            continue
        final_chunks.append({
            "content": "\n\n".join(window),
            "metadata": {
                "heading": heading,
                "paragraph_start": i,
                "paragraph_end": i + len(window) - 1
            }
        })
        if i + WINDOW_SIZE >= len(paragraphs):
            break

### Print Slide chunking (Diff)

In [ ]:
WINDOW_SIZE = 3
OVERLAP = 1
STEP = WINDOW_SIZE - OVERLAP

for heading, paragraphs in sections.items():

    if len(paragraphs) < 5:
        continue

    print("\n" + "=" * 120)
    print("HEADING:", heading)
    print("=" * 120)

    print("\nORIGINAL PARAGRAPHS\n")

    for i, para in enumerate(paragraphs[:10], 1):
        print(f"\n[P{i}]")
        print("-" * 80)
        print(para)

    print("\n\nSLIDING CHUNKS\n")

    chunk_no = 1

    for i in range(0, min(len(paragraphs), 10), STEP):

        window = paragraphs[i:i + WINDOW_SIZE]

        if not window:
            break

        print("\n" + "#" * 120)
        print(
            f"CHUNK {chunk_no} "
            f"(Paragraphs {i+1} → {i+len(window)})"
        )
        print("#" * 120)

        for j, para in enumerate(window, start=i+1):
            print(f"\n(P{j})")
            print("-" * 80)
            print(para)

        chunk_no += 1

        if i + WINDOW_SIZE >= 10:
            break

    break

## Token Validation

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

In [ ]:
for chunk in final_chunks:
    chunk["token_count"] = len(
        tokenizer.encode(
            chunk["content"],
            add_special_tokens=False
        )
    )

In [ ]:
counts = [c["token_count"] for c in final_chunks]

print("Min:", min(counts))
print("Max:", max(counts))
print("Avg:", round(sum(counts) / len(counts), 2))

In [ ]:
counts = [c["token_count"] for c in final_chunks]

print("Chunks < 50:", sum(x < 50 for x in counts))
print("Chunks 50-100:", sum(50 <= x < 100 for x in counts))
print("Chunks 100-300:", sum(100 <= x < 300 for x in counts))
print("Chunks > 300:", sum(x >= 300 for x in counts))

In [ ]:
for chunk in final_chunks[:3]:
    print("=" * 100)
    print(chunk["metadata"])
    print("TOKENS:", chunk["token_count"])
    print()
    print(chunk["content"])

# BGE (BAAI General Embedding)

## Load BGE Model

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

## Generate embeddings

In [ ]:
embeddings = model.encode(
    [chunk["content"] for chunk in final_chunks],
    normalize_embeddings=True,
    show_progress_bar=True
)

## Attach embeddings back to chunks

In [ ]:
for chunk, embedding in zip(final_chunks, embeddings):
    chunk["embedding"] = embedding.tolist()

### Verify Emebeddings

In [ ]:
print("Total chunks:", len(final_chunks))
print("Embedding dimensions:", len(final_chunks[0]["embedding"]))

## Test semantic similarity

In [ ]:
query = "What are the advantages of cloud computing?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

In [ ]:
import numpy as np

scores = [
    np.dot(query_embedding, chunk["embedding"])
    for chunk in final_chunks
]

best_idx = np.argmax(scores)

print("Score:", scores[best_idx])
print()
print(final_chunks[best_idx]["metadata"])
print()
print(final_chunks[best_idx]["content"])

## Proper Top-K retrieval

In [ ]:
query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

scores = [
    np.dot(query_embedding, chunk["embedding"])
    for chunk in final_chunks
]

top_k = 5

top_indices = np.argsort(scores)[::-1][:top_k]

for rank, idx in enumerate(top_indices, 1):
    print(f"\n{'='*100}")
    print(f"RANK {rank}")
    print(f"SCORE: {scores[idx]:.4f}")
    print(final_chunks[idx]["metadata"])
    print(f"{'='*100}")
    print(final_chunks[idx]["content"][:500])

# Dump in DataBase

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
DB_PATH = "/content/drive/MyDrive/chroma_db"
os.makedirs(DB_PATH, exist_ok=True)

## Store in DB

In [ ]:
import faiss
import numpy as np

vectors = np.array(embeddings, dtype="float32")
dimension = vectors.shape[1]
index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(vectors)
index.add(vectors)
print("Vectors stored:", index.ntotal)

## Write file in Google Drive

In [ ]:
from pathlib import Path
import faiss
index_path = "/content/drive/MyDrive/aws_rag/faiss_index.bin"
Path(index_path).parent.mkdir(
    parents=True,
    exist_ok=True
)
faiss.write_index(index, index_path)

## Save chunk metadata separately

In [ ]:
import pickle
with open(
    "/content/drive/MyDrive/aws_rag/chunks.pkl",
    "wb"
) as f:
    pickle.dump(final_chunks, f)

# Load DB from Google Drive

## Fetch file from Google Drive

In [ ]:
import faiss
import pickle
index = faiss.read_index(
    "/content/drive/MyDrive/aws_rag/faiss_index.bin"
)
with open(
    "/content/drive/MyDrive/aws_rag/chunks.pkl",
    "rb"
) as f:
    final_chunks = pickle.load(f)

## Run a sample Query

In [ ]:
query = "What is cloud computing?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

scores, ids = index.search(
    np.array([query_embedding], dtype="float32"),
    k=5
)
print("Vectors:", index.ntotal)
print("Chunks:", len(final_chunks))
print("\nSample Metadata:")
print(final_chunks[0]["metadata"])

## Print sample Retrival output

In [ ]:
for score, idx in zip(scores[0], ids[0]):
    print("=" * 100)
    print("Score:", score)
    print(final_chunks[idx]["metadata"])
    print(final_chunks[idx]["content"][:500])

# Add your Query

In [ ]:
query = "What is EC2?"

# Reranker Model

### Load Reranker Model

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

### Get candidates from FAISS

In [ ]:
candidates = [
    final_chunks[idx]
    for idx in ids[0]
]

### Create query-chunk pairs

In [ ]:
pairs = [
    (query, chunk["content"])
    for chunk in candidates
]

### Score them

In [ ]:
rerank_scores = reranker.predict(pairs)
ranked = sorted(
    zip(rerank_scores, candidates),
    key=lambda x: x[0],
    reverse=True
)

### Print Score

In [ ]:
for score, chunk in ranked[:3]:
    print("=" * 100)
    print("Rerank Score:", score)
    print(chunk["metadata"])
    print(chunk["content"][:500])

# Build Context (Meta Data Aware)

In [ ]:
def build_context(ranked_chunks, top_k=3):

    context_parts = []

    for rank, (score, chunk) in enumerate(ranked_chunks[:top_k], start=1):

        meta = chunk["metadata"]

        context_parts.append(
            f"""
Document {rank}

Heading: {meta['heading']}
Paragraphs: {meta['paragraph_start']} - {meta['paragraph_end']}
Relevance Score: {score:.4f}

Content:
{chunk['content']}
"""
        )

    return "\n" + ("\n" + "=" * 100 + "\n").join(context_parts)

In [ ]:
context = build_context(ranked)

In [ ]:
print(context)

# Prompt Building

In [ ]:
prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
print(prompt)

# LLM

## Load Model

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

## Genrate answer

In [ ]:
response = llm(
    prompt,
    max_new_tokens=200,
    do_sample=False
)

print(response[0]["generated_text"])